## 1. Load and read the dataset

Read the dataset from hdf5 file and show the data struction

In [1]:
import h5py
import json
import os
import random

def get_all_hdf5_files(directory):
    """
    Get all HDF5 files in a directory and its subdirectories.
    """
    hdf5_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.hdf5'):
                hdf5_files.append(os.path.join(root, file))
    return hdf5_files

def print_structure(group, indent=0):
    """
    Print the hierarchical structure of an HDF5 group.
    """
    for key in group.keys():
        item = group[key]
        print(" " * indent + key)
        if isinstance(item, h5py.Group):
            print_structure(item, indent + 4)
        elif isinstance(item, h5py.Dataset):
            print(" " * indent, item.shape, item.dtype)
            if key == "entities" or key == "target_entity" or key == "instruction":
                decoded = [x.decode('utf-8') for x in item]
                print(" " * indent, decoded)
        
        elif isinstance(item, h5py.Datatype):
            print(" " * (indent + 4) + "Datatype", json.loads())

In [ ]:
def get_data(group, indent=0):
    """
    Print the hierarchical structure of an HDF5 group.
    """
    data = {}
    for key in group.keys():
        item = group[key]
        if isinstance(item, h5py.Group):
            data_new = get_data(item, indent + 4)
            print(data_new)
            data.update(data_new)
        elif isinstance(item, h5py.Dataset):
            if key == "entities" or key == "target_entity" or key == "instruction":
                decoded = [x.decode('utf-8') for x in item]
                data[key] = decoded
        
        elif isinstance(item, h5py.Datatype):
            print(" " * (indent + 4) + "Datatype", json.loads())
    return data

In [3]:
dataset_root = "/workspace/robotics/home_wzr/benchmark/VLABench/dataset/dataset/select_toy" 
hdf5_files = get_all_hdf5_files(dataset_root)
print(hdf5_files[:3])

['/workspace/robotics/home_wzr/benchmark/VLABench/dataset/dataset/select_toy/data_9.hdf5', '/workspace/robotics/home_wzr/benchmark/VLABench/dataset/dataset/select_toy/data_4.hdf5', '/workspace/robotics/home_wzr/benchmark/VLABench/dataset/dataset/select_toy/data_13.hdf5']


In [ ]:
example_file = random.choice(hdf5_files)
with h5py.File(example_file, 'r') as f:
    print_structure(f)

data
    2025-04-01 15:42:59
        instruction
         (1,) |S34
         ['Put the nami into the giftbox_seen']
        meta_info
            entities
             (5,) |S12
             ['table', 'giftbox_seen', 'nami', 'hawkeye', 'slinky_dog']
            episode_config
             () |S1830
            target_entity
             (1,) |S4
             ['nami']
        observation
            depth
             (131, 4, 480, 480) float32
            ee_state
             (131, 8) float32
            point_cloud_colors
             (131, 22325, 3) float32
            point_cloud_points
             (131, 22325, 3) float32
            q_acceleration
             (131, 7, 1) float32
            q_state
             (131, 7, 1) float32
            q_velocity
             (131, 7, 1) float32
            rgb
             (131, 4, 480, 480, 3) uint8
            robot_mask
             (131, 4, 480, 480) float32
        trajectory
         (131, 8) float32


In [18]:
def extract_data(group, result, indent=0):
    """
    Recursively extract data from HDF5 group into a dictionary.
    """
    for key in group.keys():
        item = group[key]
        if isinstance(item, h5py.Group):
            # 如果是组，则递归调用
            result[key] = {}
            extract_data(item, result[key], indent + 4)
        elif isinstance(item, h5py.Dataset):
            # 如果是数据集，则提取数据
            if item.shape == ():  # 判断是否是标量数据
                # 标量数据直接读取
                result[key] = item[()]  # 读取标量 (比如标量值、字符串等)
                if isinstance(result[key], bytes):  # 如果是字节字符串，解码为正常字符串
                    result[key] = result[key].decode('utf-8')
            else:
                if item.dtype == 'S':  # 如果是字符串类型（固定长度字符串），需要解码
                    result[key] = [x.decode('utf-8') for x in item]
                else:
                    result[key] = item[:]  # 非标量数据正常切片读取
    return result

import h5py

filename = '/workspace/robotics/home_wzr/benchmark/VLABench/dataset/dataset/select_toy/data_4.hdf5'
result = {}
with h5py.File(filename, 'r') as file:
    data = extract_data(file, result)

In [40]:
import json
result = data.copy()
result = result['data']
key = list(result.keys())
result = result[key[0]]
result['trajectory'].shape

(152, 8)

## 2. Convert to tf dataset

In the baseline methods, OpenVLA and Octo are trained using datasets in the TFDS format. In this repository, we have modified the tfds dataset builder to convert the HDF5 source format of VLABench into TFDS format. We offer [singlethread tfds builder](../VLABench/utils/rlds_builder.py) and [multithread tfds builder](../VLABench/utils/multithread_rlds_builder.py).

To convert the dataset, run
```sh
python scripts/convert_to_rlds.py --save_dir /your/vlabench/dataset/root --task {target_task}
```
to create the tfds builder in the dataset directory.
Then, run
```sh
cd /your/vlabench/dataset/root/target_task
tfds build --overwrite
```
to generate rlds format dataset.

We recommend the readers to refer to https://github.com/kpertsch/rlds_dataset_builder for further guidance.

## 3. Convert to Lerobot dataset

In the baseline methods, $\pi_0$ uses lerobot format dataset for finetining. We offer a script to convert hdf5 format dataset into lerobot format, similar to Libero dataset used in $\pi_0$.

Run 
```sh
python scripts/convert_to_lerobot.py --dataset-name xxx --dataset-path /your/path/to/hdf5 --max-files 500 --task-list task1 task2 ... task n
```
to create a multi-task lerobot dataset.